# AD-Early-Prediction — Example Usage

Train, evaluate, and run a lead-time analysis for both classifiers (CN → MCI and MCI → AD).

The NACC data used in the paper are restricted (Data Use Agreement) and are **not** included in
this repository. This notebook therefore:

1. tries to load the prepared datasets (`datasets/Dataset_v2_1/*.csv`) if you have already run
   `preprocessing.run_pipeline`, and
2. otherwise falls back to a small **synthetic cohort** with the correct schema so the notebook
   runs end-to-end without any NACC data.

The example uses reduced hyperparameter budgets so it finishes in a few minutes. See the final
markdown cell for how to scale up to the paper's settings.


In [ ]:
import importlib
import os

import numpy as np
import pandas as pd

import preprocessing
import feature_engineering
import model
import leadtime
import visualization

for _m in (preprocessing, feature_engineering, model, leadtime, visualization):
    importlib.reload(_m)

from model import train_best_model
from leadtime import run_leadtime, analyze_run, _parse_list
from visualization import (
    plot_feature_importance, plot_confusion_mat, plot_roc, plot_pr_curve,
)


In [ ]:
# ── Synthetic cohort generator (schema-identical to run_pipeline output) ───────
SCALAR_SPECS = {
    "SEX": (1, 2), "EDUC": (12, 20), "NACCFAM": (0, 1),
    "CVHATT": (0, 1), "CVAFIB": (0, 1), "DIABETES": (0, 1),
    "HYPERCHO": (0, 1), "HYPERTEN": (0, 1), "B12DEF": (0, 1),
    "DEPD": (0, 1), "ANX": (0, 1), "NACCTBI": (0, 1),
    "SMOKYRS": (0, 40), "RACE": (1, 5), "HISPANIC": (0, 1),
    "NACCNE4S": (0, 2),
}
LONG_SPECS = {
    "NACCBMI": (18.0, 40.0), "NACCMMSE": (20.0, 30.0),
    "NACCGDS": (0.0, 8.0), "CDRSUM": (0.0, 8.0),
    "TOBAC30": (0, 1), "BILLS": (0, 3), "TAXES": (0, 3),
    "SHOPPING": (0, 3), "GAMES": (0, 3), "STOVE": (0, 3),
    "MEALPREP": (0, 3), "EVENTS": (0, 3), "PAYATTN": (0, 3),
    "REMDATES": (0, 3), "TRAVEL": (0, 3),
    "NACCLIVS": (1, 5), "COMMUN": (0.0, 3.0), "ALCOHOL": (0, 2),
    "hearing": (0.0, 2.0), "vision": (0.0, 2.0),
}


def make_synthetic_cohort(n_subjects, start_label, p_progressor, seed=0):
    """Build a subject-level DataFrame matching the `run_pipeline` output schema.

    start_label : 0 for CN→MCI, 1 for MCI→AD.
    """
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n_subjects):
        n_visits = int(rng.integers(5, 10))          # 5–9 visits
        is_prog = rng.random() < p_progressor

        prog = [start_label] * n_visits
        prog_id = 0
        if is_prog and n_visits >= 4:
            conv_visit = int(rng.integers(2, n_visits - 1))
            prog[conv_visit:] = [start_label + 1] * (n_visits - conv_visit)
            prog_id = 1

        months = [0.0]
        for _ in range(n_visits - 1):
            months.append(round(months[-1] + rng.uniform(10.0, 20.0), 2))

        # Progressors drift in the "worse" direction on cognition/function.
        drift = {}
        if is_prog:
            drift = {
                "NACCMMSE": -rng.uniform(1.0, 3.0),
                "CDRSUM": rng.uniform(0.5, 2.0),
                "NACCGDS": rng.uniform(0.5, 1.5),
                "COMMUN": rng.uniform(0.25, 1.0),
            }

        long_vals = {}
        for col, (lo, hi) in LONG_SPECS.items():
            if isinstance(lo, float) or isinstance(hi, float):
                vals = rng.uniform(lo, hi, n_visits)
                if col in drift:
                    vals = vals + np.linspace(0.0, drift[col], n_visits)
                vals = np.clip(vals, lo, hi)
                long_vals[col] = [float(round(v, 2)) for v in vals]
            else:
                vals = rng.integers(lo, hi + 1, n_visits).astype(float)
                if col in drift:
                    vals = vals + np.linspace(0.0, drift[col], n_visits)
                vals = np.clip(np.rint(vals), lo, hi).astype(int)
                long_vals[col] = [int(v) for v in vals]

        scalar_vals = {
            col: int(rng.integers(lo, hi + 1))
            for col, (lo, hi) in SCALAR_SPECS.items()
        }

        rows.append({
            "ID": f"SYN{i:04d}",
            "Prog_ID": prog_id,
            "Progression": tuple(prog),
            "n_visits": n_visits,
            "age": int(rng.integers(50, 91)),
            **scalar_vals,
            **long_vals,
            "months_since_baseline": months,
        })
    return pd.DataFrame(rows)


In [ ]:
# ── Load real datasets if present, otherwise synthesize ───────────────────────
CANDIDATES = [
    "datasets/Dataset_v2_1",      # run_pipeline output inside this repo
    "../datasets/Dataset_v2_1",   # full-project layout (public-repo/ subfolder)
]

def _find(rel_path):
    for base in CANDIDATES:
        full = os.path.join(base, rel_path)
        if os.path.exists(full):
            return full
    return None

cn_path = _find("CN_MCI.csv")
mci_path = _find("MCI_AD.csv")

if cn_path and mci_path:
    print("Using real datasets:")
    print("  CN_MCI:", cn_path)
    print("  MCI_AD:", mci_path)
    cn_df = pd.read_csv(cn_path)
    mci_df = pd.read_csv(mci_path)
else:
    print("No prepared datasets found — generating synthetic cohorts.")
    cn_df = make_synthetic_cohort(120, start_label=0, p_progressor=0.4, seed=0)
    mci_df = make_synthetic_cohort(120, start_label=1, p_progressor=0.4, seed=1)

lead_cn_path = _find("lead_time_CN.csv")
lead_mci_path = _find("lead_time_MCI_AD.csv")
if not (lead_cn_path and lead_mci_path):
    os.makedirs("datasets/Dataset_v2_1", exist_ok=True)
    make_synthetic_cohort(50, 0, 0.5, seed=2).to_csv(
        "datasets/Dataset_v2_1/lead_time_CN.csv", index=False)
    make_synthetic_cohort(50, 1, 0.5, seed=3).to_csv(
        "datasets/Dataset_v2_1/lead_time_MCI_AD.csv", index=False)
    lead_cn_path = "datasets/Dataset_v2_1/lead_time_CN.csv"
    lead_mci_path = "datasets/Dataset_v2_1/lead_time_MCI_AD.csv"
    print("Wrote synthetic lead-time CSVs to datasets/Dataset_v2_1/")

print("CN cohort:", cn_df.shape, "| MCI/AD cohort:", mci_df.shape)


### (Optional) Preprocessing from a raw NACC file

If you have obtained the raw NACC investigator CSV, build the subject-level datasets with:

```python
from preprocessing import run_pipeline

run_pipeline(
    source_csv="investigator_ftldlbd_nacc72.csv",
    dest_dir="datasets/Dataset_v2_1",
    min_visits=2, max_visits=10, min_age=50,
    lead_time_pct=0.05, do_impute=False,
)
```


In [ ]:
# ── Training configuration (reduced budget for a quick example) ────────────────
PARAMS = {
    "n_estimators": (50, 300),
    "max_depth": (3, 6),
    "learning_rate": (0.01, 0.3, "log"),
    "subsample": (0.5, 1.0),
    "colsample_bytree": (0.5, 1.0),
    "min_child_weight": (1, 10),
    "gamma": (0.0, 5.0),
    "reg_alpha": (1e-3, 10.0, "log"),
    "reg_lambda": (1e-3, 10.0, "log"),
}
N_TRIALS = 10

# Create output directories up front: train_best_model writes SHAP/PR charts into
# save_dir during build_model_final, before it creates save_dir itself.
os.makedirs("example_output/cn_mci", exist_ok=True)
os.makedirs("example_output/mci_ad", exist_ok=True)

cn_model, cn_cols, cn_summary = train_best_model(
    cn_df,
    progression_type="CN",
    params=PARAMS,
    csv_path="example_output/cn_mci_cv_scores.csv",
    save_dir="example_output/cn_mci",
    n_jobs=1,
    n_trials=N_TRIALS,
    objective_metric="auc",
    model_base_name="cn_mci_example",
    save_artifacts=True,
    random_state=42,
)
print("CN→MCI trained. ROC-AUC:", round(cn_summary["base_auc"], 3),
      "| PR-AUC:", round(cn_summary["base_avg_precision"], 3))

In [ ]:
mci_model, mci_cols, mci_summary = train_best_model(
    mci_df,
    progression_type="AD",
    params=PARAMS,
    csv_path="example_output/mci_ad_cv_scores.csv",
    save_dir="example_output/mci_ad",
    n_jobs=1,
    n_trials=N_TRIALS,
    objective_metric="auc",
    model_base_name="mci_ad_example",
    save_artifacts=True,
    random_state=42,
)
print("MCI→AD trained. ROC-AUC:", round(mci_summary["base_auc"], 3),
      "| PR-AUC:", round(mci_summary["base_avg_precision"], 3))


In [ ]:
# ── Quick evaluation plots for both models ────────────────────────────────────
for label, m, cols, summ, slug in [
    ("CN→MCI", cn_model, cn_cols, cn_summary, "CN_to_MCI"),
    ("MCI→AD", mci_model, mci_cols, mci_summary, "MCI_to_AD"),
]:
    print(f"\n=== {label} ===")
    plot_feature_importance(
        m.feature_importances_, cols, top_n=15,
        title=f"Top 15 features — {label}",
        save_path=f"example_output/{slug}_importance.png",
    )
    plot_confusion_mat(
        summ["y_true"], summ["y_pred"],
        title=f"Confusion matrix — {label}",
        save_path=f"example_output/{slug}_confusion.png",
    )
    plot_roc(
        summ["y_true"], summ["y_proba"],
        title=f"ROC — {label}",
        save_path=f"example_output/{slug}_roc.png",
    )
    plot_pr_curve(
        summ["y_true"], summ["y_proba"],
        title=f"Precision-Recall — {label}",
        save_path=f"example_output/{slug}_pr.png",
    )

In [ ]:
# ── Lead-time analysis at the paper's operating points ────────────────────────
# CN→MCI: threshold 0.25, mask 3   |   MCI→AD: threshold 0.30, mask 2
SPECS = {
    "CN_MCI": (cn_model, "CN", lead_cn_path, 0.25, 3),
    "MCI_AD": (mci_model, "AD", lead_mci_path, 0.30, 2),
}

for cohort, (m, prog, lead_path, thr, mask) in SPECS.items():
    dest = f"example_output/leadtime_{cohort}"
    run_leadtime(lead_path, dest, m, prog, mask_length=0)

    prog_df = pd.read_csv(os.path.join(dest, "lead_time_probabilities.csv"))
    ctrl_df = pd.read_csv(os.path.join(dest, "control_probabilities.csv"))
    for df in (prog_df, ctrl_df):
        df["months_since_baseline"] = df["months_since_baseline"].apply(_parse_list)
        df["Progression"] = df["Progression"].apply(_parse_list)

    res = analyze_run(prog_df, ctrl_df, thr, mask)
    print(f"\n=== {cohort} (threshold={thr}, mask={mask}) ===")
    print(f"  progressors: {res['n_prog']}   controls: {res['n_ctrl']}")
    print(f"  Sensitivity:      {res['sensitivity']:.3f}")
    print(f"  Specificity:      {res['specificity']:.3f}")
    print(f"  False-alarm rate: {res['false_alarm_rate']:.3f}")
    print(f"  Mean lead time (early):  {res['mean_lead_time']:.1f} months")
    print(f"  Mean net lead time:      {res['mean_net_lead_time']:.1f} months")
    print(f"  ROC-AUC (subject-level): {res['roc_auc']:.3f}")


## Scaling up to the paper's settings

The paper uses much larger budgets than this example:

- `n_trials=1000` (instead of `10`).
- **Non-imputed:** 10 random stratified train/test splits (seeds from a master seed of 42).
- **MICE-imputed:** 10 imputation variants × 10 stratified-bootstrap lead-time draws, with
  `fit_imputer` fitted on the training split only and applied via `transform_imputer`.
- The full loops live in `leadtime_analysis.ipynb`; the SHAP summary and PR curves are
  generated automatically during training and saved under `example_output/<name>/`.

Generated artifacts live in `example_output/` (models, reports, charts, CV scores, and the
lead-time probability cache).
